# SentiSign v2 — WLASL100 on Kaggle GPU
**Setup (once):** right panel → *Add Input* → search `wlasl-processed` (by risangbaskoro) → Add. *Settings* → Accelerator **GPU T4** → Internet **On**. Then **Run All**. At the end, download `sentisign_reports.zip` from the Output panel.

In [ ]:
!pip -q install mediapipe opencv-python-headless onnx
import torch, os; print(torch.__version__, torch.cuda.is_available(), torch.cuda.get_device_name(0) if torch.cuda.is_available() else '')
os.makedirs('/kaggle/working/sv2', exist_ok=True); os.chdir('/kaggle/working/sv2')

In [ ]:
import os
os.makedirs('sentisign', exist_ok=True); os.makedirs('scripts', exist_ok=True)
open('sentisign/__init__.py','w').close(); open('scripts/__init__.py','w').close()

In [ ]:
%%writefile sentisign/calibration.py
import numpy as np


def softmax(z): z = z - z.max(1, keepdims=True); e = np.exp(z); return e / e.sum(1, keepdims=True)


def fit_temperature(logits, y, grid=np.concatenate([np.linspace(0.1, 5, 99), np.linspace(5.5, 30, 50)])):
    best, bt = 1e9, 1.0
    for t in grid:
        p = softmax(logits / t); nll = -np.log(p[np.arange(len(y)), y] + 1e-12).mean()
        if nll < best: best, bt = nll, t
    return bt


def ece(probs, y, bins=15):
    conf = probs.max(1); pred = probs.argmax(1); acc = (pred == y)
    edges = np.linspace(0, 1, bins + 1); e = 0.0
    for lo, hi in zip(edges[:-1], edges[1:]):
        m = (conf > lo) & (conf <= hi)
        if m.any(): e += m.mean() * abs(acc[m].mean() - conf[m].mean())
    return float(e)


def abstain_threshold(probs, y, target_acc=0.9):
    """Smallest confidence threshold whose accepted-set accuracy >= target."""
    conf = probs.max(1); acc = probs.argmax(1) == y
    for t in np.linspace(0, 0.99, 100):
        m = conf >= t
        if m.sum() >= 10 and acc[m].mean() >= target_acc: return float(t), float(m.mean())
    return 1.0, 0.0


def topk_fallback(probs, y, thr, k=3):
    """When the model abstains, show top-k choices; returns how often the right sign is among them."""
    low = probs.max(1) < thr
    if not low.any(): return None
    topk = np.argsort(-probs[low], 1)[:, :k]
    return float(np.mean([y[low][i] in topk[i] for i in range(low.sum())]))


In [ ]:
%%writefile sentisign/emotion.py
"""Emotion from face blendshapes (52-d, MediaPipe). Synthetic here; real data on Colab.
Per-user neutral calibration: subtract the user's mean neutral vector.
Grammar confound: raised brows in questions (browInnerUp) mimic Surprise."""
import numpy as np
EMOS = ["Neutral", "Happy", "Sad", "Angry", "Fear", "Disgust", "Surprise"]
D = 52
AFFECT_IDX = np.arange(0, 40)       # affect-related blendshapes
GRAMMAR_IDX = np.arange(40, 52)     # brow/head cues used for questions/negation


def make_users(n, seed=0):
    r = np.random.default_rng(seed); return [dict(rest=r.normal(size=D) * 0.25, gain=r.uniform(.6, 1.4)) for _ in range(n)]


def proto(seed=1):
    r = np.random.default_rng(seed); p = np.zeros((7, D)); p[1:, :40] = r.normal(size=(6, 40)) * 0.28; return p


def sample(users, protos, n_per=40, grammar_p=0.25, seed=2):
    r = np.random.default_rng(seed); X, y, u = [], [], []
    for ui, us in enumerate(users):
        for e in range(7):
            for _ in range(n_per):
                x = us["rest"] + us["gain"] * protos[e] + r.normal(size=D) * 0.55
                if r.random() < grammar_p:  # question brows: grammar cue AND leaks into Surprise-like affect dims
                    x[GRAMMAR_IDX] += r.normal(1.2, .3, len(GRAMMAR_IDX)); x[:40] += 0.8 * protos[6, :40] * us['gain']
                X.append(x); y.append(e); u.append(ui)
    return np.array(X), np.array(y), np.array(u)


def calibrate(X, u, y, use=True):
    """Subtract each user's mean over their Neutral frames (what a 3-second calibration captures)."""
    if not use: return X.copy()
    Z = X.copy()
    for ui in np.unique(u):
        m = u == ui; Z[m] -= X[m & (y == 0)].mean(0)
    return Z


In [ ]:
%%writefile sentisign/endpoint.py
"""Endpointing: sign segment ends when both hands are 'at rest' for rest_s seconds."""
import numpy as np


def hand_speed(loc_seq):
    """loc_seq (T,42) body-frame location; speed = |delta wrist| for each hand."""
    d = np.diff(loc_seq, axis=0, prepend=loc_seq[:1])
    return np.linalg.norm(d[:, :3], axis=1) + np.linalg.norm(d[:, 21:24], axis=1)


class Endpointer:
    def __init__(self, fps=30, rest_s=0.8, speed_thr=0.004, min_active_s=0.3):
        self.rest_n = int(rest_s * fps); self.thr = speed_thr
        self.min_active = int(min_active_s * fps)
        self.reset()

    def reset(self):
        self.active = 0; self.rest = 0; self.prev = None; self.buf = []

    def push(self, loc, present=True):
        """Returns segment (list of frames) when an utterance ends, else None."""
        sp = 0.0 if self.prev is None or not present else float(np.abs(loc[[0,1,2,21,22,23]] - self.prev[[0,1,2,21,22,23]]).sum())
        self.prev = loc
        moving = present and sp > self.thr
        if moving:
            self.active += 1; self.rest = 0
        elif self.active:
            self.rest += 1
        if self.active:
            self.buf.append(loc)
        if self.active >= self.min_active and self.rest >= self.rest_n:
            seg = self.buf[:-self.rest_n] if self.rest_n else self.buf
            self.reset()
            return seg
        return None


In [ ]:
%%writefile sentisign/features.py
"""Multi-stream sign features. Pure numpy so Python and browser (JS port) can be
checked against the same golden vectors.

Input per frame: hands (2,21,3) [L,R], pose (33,3) MediaPipe layout, face anchors (5,3)
(nose, chin, forehead, left cheek, right cheek) with NaN where missing.
Body frame: origin = shoulder midpoint, scale = shoulder width, so location is
signer-size independent and hand position relative to face/torso is preserved.
"""
import numpy as np

L_SH, R_SH = 11, 12
EPS = 1e-6


def body_frame(pose):
    ls, rs = pose[L_SH], pose[R_SH]
    origin = (ls + rs) / 2.0
    scale = np.linalg.norm(ls[:2] - rs[:2])
    return origin, max(scale, EPS)


def handshape(hand):
    """63-d wrist-relative, hand-size-normalised shape (location removed)."""
    if np.isnan(hand).any():
        return np.zeros(63, dtype=np.float32)
    rel = hand - hand[0]
    size = np.linalg.norm(hand[9] - hand[0]) + EPS
    return (rel / size).reshape(-1).astype(np.float32)


def location(hand, origin, scale, anchors):
    """Wrist + index-tip position in body frame (6) + wrist-to-anchor vectors (15)."""
    if np.isnan(hand).any():
        return np.zeros(21, dtype=np.float32)
    w = (hand[0] - origin) / scale
    t = (hand[8] - origin) / scale
    parts = [w, t]
    for a in anchors:
        parts.append(np.zeros(3) if np.isnan(a).any() else (hand[0] - a) / scale)
    return np.concatenate(parts).astype(np.float32)


def orientation(hand):
    """Palm normal (3) from wrist, index MCP, pinky MCP."""
    if np.isnan(hand).any():
        return np.zeros(3, dtype=np.float32)
    n = np.cross(hand[5] - hand[0], hand[17] - hand[0])
    return (n / (np.linalg.norm(n) + EPS)).astype(np.float32)


def frame_features(hands, pose, anchors):
    origin, scale = body_frame(pose)
    hs, loc, ori = [], [], []
    for h in hands:
        hs.append(handshape(h)); loc.append(location(h, origin, scale, anchors[:5])); ori.append(orientation(h))
    return np.concatenate(hs), np.concatenate(loc), np.concatenate(ori)


def fill_missing_hands(hands_seq, max_gap=15):
    """Linearly interpolate a hand across short MediaPipe dropouts (<= max_gap frames); edges are
    held from the nearest detection. A hand never seen stays NaN (-> zeros downstream)."""
    H = np.array(hands_seq, dtype=float).copy(); n = len(H)
    for k in range(H.shape[1]):
        ok = ~np.isnan(H[:, k]).any(axis=(1, 2))
        if ok.all() or not ok.any(): continue
        idx = np.where(ok)[0]
        for i in np.where(~ok)[0]:
            lo = idx[idx < i]; hi = idx[idx > i]
            if len(lo) and len(hi):
                if hi[0] - lo[-1] - 1 > max_gap: continue
                w = (i - lo[-1]) / (hi[0] - lo[-1]); H[i, k] = (1 - w) * H[lo[-1], k] + w * H[hi[0], k]
            elif len(lo) and i - lo[-1] <= max_gap: H[i, k] = H[lo[-1], k]
            elif len(hi) and hi[0] - i <= max_gap: H[i, k] = H[hi[0], k]
    return H


def sequence_features(hands_seq, pose_seq, anchor_seq, T=60, fill=True):
    """Returns dict of streams, each (T, d). Movement = frame-to-frame delta of location."""
    if fill: hands_seq = fill_missing_hands(hands_seq)
    n = len(hands_seq)
    H, Lc, O = [], [], []
    for i in range(n):
        h, l, o = frame_features(hands_seq[i], pose_seq[i], anchor_seq[i])
        H.append(h); Lc.append(l); O.append(o)
    H, Lc, O = map(np.array, (H, Lc, O))
    idx = np.linspace(0, n - 1, T).round().astype(int)  # resample to fixed length
    H, Lc, O = H[idx], Lc[idx], O[idx]
    M = np.diff(Lc, axis=0, prepend=Lc[:1])
    return {"shape": H, "loc": Lc, "move": M, "orient": O}


STREAM_DIMS = {"shape": 126, "loc": 42, "move": 42, "orient": 6}
TOTAL_DIM = sum(STREAM_DIMS.values())  # 216


def concat(streams):
    return np.concatenate([streams[k] for k in ("shape", "loc", "move", "orient")], axis=-1)


In [ ]:
%%writefile sentisign/language.py
"""Rule baseline gloss->English (what a fine-tuned Flan-T5 must beat) + TTS emotion map."""
import re
TTS = {"Happy": (0.65, 0.25), "Angry": (0.82, 0.68), "Sad": (0.22, 0.88), "Fear": (0.68, 0.82),
       "Disgust": (0.55, 0.52), "Surprise": (0.78, 0.22), "Neutral": (0.38, 0.32)}  # (exaggeration, cfg)
PRON = {"ME": "I", "I": "I", "YOU": "you", "HE": "he", "SHE": "she", "WE": "we", "THEY": "they"}
VERB = {"WANT", "LIKE", "NEED", "HAVE", "GO", "EAT", "DRINK", "HELP", "LOVE", "KNOW", "SEE", "WORK", "LEARN"}
Q = {"WHAT", "WHERE", "WHO", "WHY", "HOW", "WHEN"}


def baseline(glosses, question=False):
    g = [x.upper() for x in glosses]
    if not g: return ""
    out = []
    for i, w in enumerate(g):
        if w in PRON: out.append(PRON[w])
        elif w == "NOT": out.append("don't" if any(x in VERB for x in g) else "not")
        else: out.append(w.lower())
    s = " ".join(out); s = s[0].upper() + s[1:]
    if question or g[0] in Q or g[-1] in Q: return s + "?"
    return s + "."


def tts_params(emotion): return TTS.get(emotion, TTS["Neutral"])


In [ ]:
%%writefile sentisign/model.py
import torch, torch.nn as nn
from .features import STREAM_DIMS


class TCNBlock(nn.Module):
    def __init__(s, c, d, p=0.2):
        super().__init__()
        s.net = nn.Sequential(nn.Conv1d(c, c, 3, padding=d, dilation=d), nn.BatchNorm1d(c), nn.GELU(), nn.Dropout(p))
    def forward(s, x): return x + s.net(x)


class StreamEncoder(nn.Module):
    def __init__(s, d_in, c=64):
        super().__init__(); s.proj = nn.Conv1d(d_in, c, 1)
        s.tcn = nn.Sequential(*[TCNBlock(c, d) for d in (1, 2, 4)])
    def forward(s, x): return s.tcn(s.proj(x.transpose(1, 2))).transpose(1, 2)  # (B,T,c)


class MultiStreamSign(nn.Module):
    """Per-stream TCN -> concat -> BiLSTM -> attention pooling -> classifier."""
    def __init__(s, n_cls, c=64, hid=128, streams=STREAM_DIMS):
        super().__init__()
        s.names = list(streams); s.dims = list(streams.values())
        s.enc = nn.ModuleList([StreamEncoder(d, c) for d in s.dims])
        s.lstm = nn.LSTM(c * len(s.dims), hid, 2, batch_first=True, bidirectional=True, dropout=0.2)
        s.att = nn.Linear(2 * hid, 1)
        s.head = nn.Sequential(nn.Dropout(0.3), nn.Linear(2 * hid, 256), nn.GELU(), nn.Linear(256, n_cls))
    def forward(s, x):
        parts = torch.split(x, s.dims, dim=-1)
        h = torch.cat([e(p) for e, p in zip(s.enc, parts)], -1)
        h, _ = s.lstm(h)
        w = torch.softmax(s.att(h), 1)
        return s.head((w * h).sum(1))


def augment(x, p=0.8):
    """Time-warp-free cheap aug: gaussian noise + random scale of location stream + temporal shift."""
    if torch.rand(1) > p: return x
    x = x + torch.randn_like(x) * 0.01
    sh = int(torch.randint(-3, 4, (1,)))
    return torch.roll(x, sh, 1)


In [ ]:
%%writefile sentisign/splits.py
import numpy as np


def signer_independent_split(signers, val_frac=0.15, test_frac=0.2, seed=0):
    """Disjoint signers across train/val/test. Raises if overlap."""
    u = np.unique(signers); r = np.random.default_rng(seed); r.shuffle(u)
    nt = max(1, int(round(len(u) * test_frac))); nv = max(1, int(round(len(u) * val_frac)))
    te, va, tr = set(u[:nt]), set(u[nt:nt+nv]), set(u[nt+nv:])
    assert not (te & va or te & tr or va & tr)
    m = lambda S: np.where(np.isin(signers, list(S)))[0]
    return m(tr), m(va), m(te)


In [ ]:
%%writefile sentisign/synth.py
"""Synthetic sign generator for pipeline smoke tests. NOT real data.
Each gloss = (handshape template, body-anchor location, movement path). Each signer adds
personal offset/scale/speed/noise so signer-independent splits are meaningful.
Body-relative glosses share handshape but differ in location (tests the location stream)."""
import numpy as np


def _rng(seed): return np.random.default_rng(seed)


def make_gloss_bank(n_gloss, seed=0, n_shapes=12):
    r = _rng(seed)
    shapes = r.normal(size=(n_shapes, 21, 3)) * 0.3
    bank = []
    for g in range(n_gloss):
        bank.append(dict(
            shape=shapes[g % n_shapes],            # many glosses share a handshape
            loc=r.uniform(-1, 1, 3) * [0.8, 1.0, 0.2],   # body-frame target (x,y,z)
            path=r.normal(size=3) * 0.35,          # movement direction
            orient=r.normal(size=3)))
    return bank


def make_signer(seed, var=1.0):
    """var scales between-signer variation (style: offset of signing space, speed, hand shape idiosyncrasy)."""
    r = _rng(10_000 + seed)
    return dict(off=r.normal(size=3) * 0.12 * var, scale=r.uniform(0.8, 1.25), speed=r.uniform(1 - .2*var, 1 + .25*var),
                tremor=r.uniform(0.01, 0.05), cam=r.normal(size=3) * 0.2, handsz=r.uniform(0.8, 1.2),
                shape_dev=r.normal(size=(21, 3)) * 0.06 * var, path_gain=r.uniform(1 - .3*var, 1 + .3*var) if var>0 else 1.0)


def render(gloss, signer, r, T=45, noise=1.0, drop=0.0):
    """Returns hands (T,2,21,3), pose (T,33,3), anchors (T,5,3) in 'camera' coords."""
    sh_w = 0.4 * signer["scale"]
    pose = np.zeros((33, 3)); pose[11] = [-sh_w/2, 0, 0]; pose[12] = [sh_w/2, 0, 0]
    pose = pose + signer["cam"]
    origin = (pose[11] + pose[12]) / 2
    face = origin + np.array([0, -0.5, 0]) * sh_w * 2
    anchors = np.stack([face, face + [0, .1*sh_w*2, 0], face + [0, -.1*sh_w*2, 0], face+[-.08*sh_w*2,0,0], face+[.08*sh_w*2,0,0]])
    t = np.linspace(0, 1, T)[:, None]
    ts = np.clip(t * signer["speed"], 0, 1)
    pos = origin + (gloss["loc"] + signer["off"]) * sh_w + gloss["path"] * signer.get("path_gain", 1) * sh_w * (ts - .5)
    pos = pos + r.normal(size=pos.shape) * signer["tremor"] * sh_w * noise
    hands = np.full((T, 2, 21, 3), np.nan)
    shape = (gloss["shape"] + signer.get("shape_dev", 0)) * signer["handsz"] * 0.08
    hand = shape + r.normal(size=shape.shape) * 0.004 * signer["handsz"] * noise
    hand = hand - hand[0]
    for i in range(T):
        if r.random() < drop: continue  # MediaPipe missed the hand this frame
        hands[i, 1] = hand + pos[i] + r.normal(size=(21, 3)) * 0.002 * noise
    return hands, np.repeat(pose[None], T, 0), np.repeat(anchors[None], T, 0)


def make_dataset(n_gloss=50, n_signers=12, reps=6, T=60, seed=0, var=1.0, noise=1.0, drop=0.0, fill=True):
    from .features import sequence_features, concat
    bank = make_gloss_bank(n_gloss, seed)
    signers = [make_signer(s, var) for s in range(n_signers)]
    r = _rng(seed + 1)
    X, y, sg = [], [], []
    for si, s in enumerate(signers):
        for g, gl in enumerate(bank):
            for _ in range(reps):
                h, p, a = render(gl, s, r, noise=noise, drop=drop)
                X.append(concat(sequence_features(h, p, a, T=T, fill=fill))); y.append(g); sg.append(si)
    return np.array(X, dtype=np.float32), np.array(y), np.array(sg)


In [ ]:
%%writefile scripts/extract_wlasl.py
"""Extract per-frame hand + pose landmarks from WLASL videos with MediaPipe Tasks (VIDEO mode).
Face anchors come from pose landmarks (nose, mouth, eyes, ears) so no face model is needed for signs.
Writes data/npz/<video_id>.npz (hands (T,2,21,3) [L,R], pose (T,33,3), anchors (T,5,3)) and
data/index_<N>.csv (path,gloss,signer,split). Resumable: skips videos already extracted.
Usage: python scripts/extract_wlasl.py --root data/wlasl --subset 100 --workers 6"""
import argparse, json, os, sys, csv, time, urllib.request
from multiprocessing import Pool
import numpy as np

HAND_URL = "https://storage.googleapis.com/mediapipe-models/hand_landmarker/hand_landmarker/float16/latest/hand_landmarker.task"
POSE_URL = "https://storage.googleapis.com/mediapipe-models/pose_landmarker/pose_landmarker_lite/float16/latest/pose_landmarker_lite.task"
MODELS = "models"


def fetch_models():
    os.makedirs(MODELS, exist_ok=True)
    for url in (HAND_URL, POSE_URL):
        p = os.path.join(MODELS, url.rsplit("/", 1)[-1])
        if not os.path.exists(p):
            print("downloading", p, flush=True); urllib.request.urlretrieve(url, p)


def anchors_from_pose(pose):
    """nose, chin proxy (mouth centre pushed down), forehead proxy (eyes pushed up), L ear, R ear."""
    nose, eyes = pose[0], (pose[2] + pose[5]) / 2
    mouth = (pose[9] + pose[10]) / 2
    chin = mouth + (mouth - nose) * 0.8
    forehead = eyes + (eyes - mouth) * 0.8
    return np.stack([nose, chin, forehead, pose[7], pose[8]])


_hl = _pl = None
def _init():
    global _hl, _pl
    import mediapipe as mp
    from mediapipe.tasks.python import vision, BaseOptions
    _hl = vision.HandLandmarker.create_from_options(vision.HandLandmarkerOptions(
        base_options=BaseOptions(model_asset_path=os.path.join(MODELS, "hand_landmarker.task")),
        running_mode=vision.RunningMode.VIDEO, num_hands=2))
    _pl = vision.PoseLandmarker.create_from_options(vision.PoseLandmarkerOptions(
        base_options=BaseOptions(model_asset_path=os.path.join(MODELS, "pose_landmarker_lite.task")),
        running_mode=vision.RunningMode.VIDEO))


def extract(job):
    vid_path, out_path, f0, f1 = job
    if os.path.exists(out_path): return out_path, "skip"
    import cv2, mediapipe as mp
    global _hl, _pl
    _init()  # fresh landmarkers per video: VIDEO mode needs increasing timestamps
    cap = cv2.VideoCapture(vid_path); fps = cap.get(cv2.CAP_PROP_FPS) or 25
    H, P, A = [], [], []; i = 0
    while True:
        ok, fr = cap.read()
        if not ok: break
        i += 1
        if i < f0 or (f1 > 0 and i > f1): continue
        h, w = fr.shape[:2]; sx = w / h   # make x isotropic with y
        img = mp.Image(image_format=mp.ImageFormat.SRGB, data=cv2.cvtColor(fr, cv2.COLOR_BGR2RGB))
        ts = int(i * 1000 / fps)
        hr = _hl.detect_for_video(img, ts); pr = _pl.detect_for_video(img, ts)
        hands = np.full((2, 21, 3), np.nan, np.float32)
        for lm, hd in zip(hr.hand_landmarks, hr.handedness):
            hands[0 if hd[0].category_name == "Left" else 1] = [[q.x * sx, q.y, q.z * sx] for q in lm]
        if pr.pose_landmarks:
            pose = np.array([[q.x * sx, q.y, q.z * sx] for q in pr.pose_landmarks[0]], np.float32)
        else:
            pose = np.full((33, 3), np.nan, np.float32)
        H.append(hands); P.append(pose); A.append(anchors_from_pose(pose))
    cap.release(); _hl.close(); _pl.close()
    if len(H) < 8: return out_path, "too_short"
    P = np.array(P)
    ok = ~np.isnan(P[:, 11, 0])
    if ok.sum() == 0: return out_path, "no_pose"
    for t in range(len(P)):        # fill missing pose frames from nearest detected frame
        if not ok[t]:
            j = np.where(ok)[0][np.argmin(np.abs(np.where(ok)[0] - t))]; P[t] = P[j]; A[t] = anchors_from_pose(P[j])
    np.savez_compressed(out_path, hands=np.array(H), pose=P, anchors=np.array(A))
    return out_path, "ok"


def main():
    ap = argparse.ArgumentParser()
    ap.add_argument("--root", default="data/wlasl"); ap.add_argument("--subset", type=int, default=100)
    ap.add_argument("--workers", type=int, default=max(1, (os.cpu_count() or 4) - 2))
    a = ap.parse_args()
    fetch_models()
    meta = json.load(open(os.path.join(a.root, "WLASL_v0.3.json")))
    glosses = [e["gloss"] for e in meta][:a.subset]   # WLASL-N = first N glosses of the json
    vids = {}
    for root, _, files in os.walk(a.root):
        for f in files:
            if f.endswith(".mp4"): vids[f[:-4]] = os.path.join(root, f)
    os.makedirs("data/npz", exist_ok=True)
    jobs, rows, missing = [], [], 0
    for e in meta[:a.subset]:
        for inst in e["instances"]:
            v = vids.get(inst["video_id"])
            if v is None: missing += 1; continue
            out = os.path.join("data/npz", inst["video_id"] + ".npz")
            jobs.append((v, out, inst.get("frame_start", 1), inst.get("frame_end", -1)))
            rows.append(dict(path=out, gloss=e["gloss"], signer=inst["signer_id"], split=inst.get("split", "")))
    print(f"WLASL{a.subset}: {len(jobs)} videos found, {missing} listed but missing; workers={a.workers}", flush=True)
    t0, stats = time.time(), {}
    with Pool(a.workers) as pool:
        for k, (out, st) in enumerate(pool.imap_unordered(extract, jobs, chunksize=4), 1):
            stats[st] = stats.get(st, 0) + 1
            if k % 50 == 0 or k == len(jobs):
                el = time.time() - t0
                print(f"  {k}/{len(jobs)} {stats} {el/60:.1f} min elapsed, ~{el/k*(len(jobs)-k)/60:.1f} min left", flush=True)
    good = [r for r in rows if os.path.exists(r["path"])]
    with open(f"data/index_{a.subset}.csv", "w", newline="") as f:
        w = csv.DictWriter(f, fieldnames=["path", "gloss", "signer", "split"]); w.writeheader(); w.writerows(good)
    print(f"index written: {len(good)} usable videos, {len(set(r['gloss'] for r in good))} glosses, "
          f"{len(set(r['signer'] for r in good))} signers", flush=True)


if __name__ == "__main__":
    main()


In [ ]:
%%writefile scripts/train_wlasl.py
"""Train + evaluate the multi-stream TCN+BiLSTM+attention model on real WLASL landmarks.
--split official : WLASL's own train/val/test (signers overlap; comparable to published numbers and v1)
--split signer   : signer-independent (no signer in more than one of train/val/test)
--ablate loc,move: drop the body-frame location and movement streams (v1-style features)
Writes reports/<name>.json and reports/<name>.onnx."""
import sys, os, json, csv, time, argparse
sys.path.insert(0, os.path.dirname(os.path.dirname(os.path.abspath(__file__))))
import numpy as np, torch, torch.nn as nn
from sentisign.features import sequence_features, concat, STREAM_DIMS
from sentisign.splits import signer_independent_split
from sentisign.model import MultiStreamSign
from sentisign.calibration import softmax, fit_temperature, ece, abstain_threshold, topk_fallback

ap = argparse.ArgumentParser()
ap.add_argument("--subset", type=int, default=100); ap.add_argument("--split", default="signer", choices=["signer", "official", "random"])
ap.add_argument("--seed", type=int, default=0); ap.add_argument("--epochs", type=int, default=150)
ap.add_argument("--ablate", default=""); ap.add_argument("--name", required=True)
a = ap.parse_args()
torch.manual_seed(a.seed); np.random.seed(a.seed)
dev = "cuda" if torch.cuda.is_available() else "cpu"
os.makedirs("reports", exist_ok=True)

cache = f"data/features_{a.subset}.npz"
if not os.path.exists(cache):
    rows = list(csv.DictReader(open(f"data/index_{a.subset}.csv")))
    X, G, S, SP = [], [], [], []
    for r in rows:
        z = np.load(r["path"])
        if np.isnan(z["hands"]).all(): continue
        X.append(concat(sequence_features(z["hands"], z["pose"], z["anchors"]))); G.append(r["gloss"]); S.append(r["signer"]); SP.append(r["split"])
    np.savez_compressed(cache, X=np.array(X, np.float32), G=np.array(G), S=np.array(S), SP=np.array(SP))
z = np.load(cache); X, G, S, SP = z["X"], z["G"], z["S"], z["SP"]
X = np.nan_to_num(X)
gl = sorted(set(G)); y = np.array([gl.index(g) for g in G])
a.ablate = a.ablate.replace("+", ",")
if a.ablate:
    off = 0
    for k, d in STREAM_DIMS.items():
        if k in a.ablate.split(","): X[..., off:off+d] = 0
        off += d
if a.split == "signer":
    tr, va, te = signer_independent_split(S, val_frac=0.15, test_frac=0.2, seed=a.seed)
elif a.split == "official":
    tr, va, te = [np.where(SP == s)[0] for s in ("train", "val", "test")]
else:
    p = np.random.default_rng(a.seed).permutation(len(y)); n = len(y)
    tr, va, te = p[:int(.7*n)], p[int(.7*n):int(.8*n)], p[int(.8*n):]
mu, sd = X[tr].mean((0, 1)), X[tr].std((0, 1)) + 1e-6; Xn = ((X - mu) / sd).astype(np.float32)
T = lambda i: (torch.tensor(Xn[i]), torch.tensor(y[i]))
Xtr, ytr = T(tr); Xva, yva = T(va); Xte, yte = T(te)

def aug(x):
    x = x + torch.randn_like(x) * 0.05
    x = torch.roll(x, int(torch.randint(-4, 5, (1,))), 1)
    m = (torch.rand(x.shape[0], x.shape[1], 1, device=x.device) < 0.1).float()   # frame dropout
    x = x * (1 - m)
    s = 1 + 0.1 * torch.randn(x.shape[0], 1, x.shape[2], device=x.device)       # per-feature jitter
    return x * s

m = MultiStreamSign(len(gl)).to(dev)
opt = torch.optim.AdamW(m.parameters(), 1e-3, weight_decay=5e-2)
steps = a.epochs * (len(tr) // 32 + 1)
sch = torch.optim.lr_scheduler.OneCycleLR(opt, 1e-3, total_steps=steps)
lf = nn.CrossEntropyLoss(label_smoothing=0.1)
best, bs, bad, t0 = -1, None, 0, time.time()
for ep in range(a.epochs):
    m.train(); perm = torch.randperm(len(Xtr))
    for i in range(0, len(perm), 32):
        b = perm[i:i+32]; xb = aug(Xtr[b].to(dev))
        opt.zero_grad(); lf(m(xb), ytr[b].to(dev)).backward(); opt.step(); sch.step()
    m.eval()
    with torch.no_grad(): acc = (m(Xva.to(dev)).argmax(1).cpu() == yva).float().mean().item()
    if acc > best: best, bs, bad = acc, {k: v.clone() for k, v in m.state_dict().items()}, 0
    else: bad += 1
    if ep % 10 == 0: print(f"  ep {ep} val {acc:.3f} best {best:.3f}", flush=True)
    if bad >= 30: break
m.load_state_dict(bs); m.eval()
with torch.no_grad(): lv, lt = m(Xva.to(dev)).cpu().numpy(), m(Xte.to(dev)).cpu().numpy()
Tm = fit_temperature(lv, y[va]); pt = softmax(lt / Tm); pr = softmax(lt)
thr, _ = abstain_threshold(softmax(lv / Tm), y[va], 0.8); acc_ = pt.argmax(1) == y[te]; mk = pt.max(1) >= thr
res = dict(name=a.name, subset=a.subset, split=a.split, seed=a.seed, ablate=a.ablate, device=dev,
           gpu=torch.cuda.get_device_name(0) if dev == "cuda" else None,
           n_classes=len(gl), n_train=len(tr), n_val=len(va), n_test=len(te),
           n_signers_train=len(set(S[tr])), n_signers_test=len(set(S[te])),
           top1=float(acc_.mean()), top5=float(np.mean([y[te][i] in np.argsort(-pt[i])[:5] for i in range(len(pt))])),
           val_best=best, ece_raw=ece(pr, y[te]), ece_cal=ece(pt, y[te]), temperature=float(Tm),
           abstain_thr_for_80pct=thr, coverage=float(mk.mean()), acc_on_accepted=float(acc_[mk].mean()) if mk.any() else None,
           top3_when_abstaining=topk_fallback(pt, y[te], thr, 3), epochs_run=ep + 1, train_min=(time.time() - t0) / 60)
json.dump(res, open(f"reports/{a.name}.json", "w"), indent=1)
print(json.dumps(res), flush=True)
torch.onnx.export(m.cpu(), torch.zeros(1, 60, X.shape[-1]), f"reports/{a.name}.onnx", opset_version=17,
                  input_names=["x"], output_names=["logits"], dynamo=False)


In [ ]:
%%writefile scripts/summarize.py
import json, glob
rows = [json.load(open(f)) for f in sorted(glob.glob("reports/*.json"))]
print(f"{'run':34s} {'split':8s} {'top1':>6s} {'top5':>6s} {'ECE':>6s} {'cov@80':>6s} {'test signers':>12s}")
for r in rows:
    print(f"{r['name']:34s} {r['split']:8s} {r['top1']*100:6.1f} {r['top5']*100:6.1f} {r['ece_cal']:6.3f} {r['coverage']*100:6.1f} {r['n_signers_test']:12d}")


In [ ]:
import glob
root = [p for p in glob.glob('/kaggle/input/**/WLASL_v0.3.json', recursive=True)][0].rsplit('/',1)[0]
print('WLASL at', root)
!python -u scripts/extract_wlasl.py --root "{root}" --subset 100 --workers 4

In [ ]:
runs = [('official',0,'wlasl100_official_s0',''),('signer',0,'wlasl100_signer_s0',''),('signer',1,'wlasl100_signer_s1',''),('signer',2,'wlasl100_signer_s2',''),('random',0,'wlasl100_random_s0',''),('signer',0,'wlasl100_signer_noloc_s0','loc+move')]
for sp, sd, name, abl in runs:
    extra = f'--ablate {abl}' if abl else ''
    !python -u scripts/train_wlasl.py --subset 100 --split {sp} --seed {sd} --name {name} {extra} 2>&1 | grep -v Warning | tail -3

In [ ]:
!python scripts/summarize.py | tee reports/summary.txt
!cd /kaggle/working && zip -qr sentisign_reports.zip sv2/reports && ls -la sentisign_reports.zip